# Sales Win-Rate Prediction — Exploratory Data Analysis

This notebook explores the 50K simulated CRM dataset before model training.

**Contents:**
1. Dataset overview
2. Target distribution
3. Win rate by category
4. Numerical feature distributions
5. Correlation heatmap
6. Key insights

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

from data.generate_data import generate_crm_data
df = generate_crm_data()
print(df.shape)
df.head()

## 1. Dataset Overview

In [ ]:
df.info()
df.describe()

## 2. Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['won'].value_counts()
axes[0].bar(['Lost', 'Won'], counts.values, color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Deal Outcomes')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['Lost', 'Won'], autopct='%1.1f%%',
            colors=['#e74c3c', '#2ecc71'], startangle=90)
axes[1].set_title(f'Win Rate: {df["won"].mean():.1%}')

plt.tight_layout()
plt.show()

## 3. Win Rate by Category

In [ ]:
cat_cols = ['industry', 'region', 'lead_source', 'company_size']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.flat, cat_cols):
    wr = df.groupby(col)['won'].mean().sort_values(ascending=False)
    bars = ax.bar(wr.index, wr.values, color=sns.color_palette('muted', len(wr)))
    ax.set_title(f'Win Rate by {col.replace("_", " ").title()}')
    ax.set_ylabel('Win Rate')
    ax.set_ylim(0, 0.8)
    ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, wr.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
                f'{val:.1%}', ha='center', fontsize=9)

plt.suptitle('Win Rate by Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Numerical Feature Distributions

In [ ]:
num_cols = ['deal_size_usd', 'sales_cycle_days', 'num_meetings',
            'num_emails', 'num_calls', 'crm_score',
            'rep_quota_attainment', 'discount_pct']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, col in zip(axes.flat, num_cols):
    won_data = df[df['won'] == 1][col]
    lost_data = df[df['won'] == 0][col]
    ax.hist(lost_data, bins=30, alpha=0.5, color='#e74c3c', label='Lost', density=True)
    ax.hist(won_data, bins=30, alpha=0.5, color='#2ecc71', label='Won', density=True)
    ax.set_title(col.replace('_', ' ').title())
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Outcome', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

In [ ]:
num_df = df[num_cols + ['won']]
corr = num_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Correlation Matrix (Numerical Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Key Insights

- **Demo & Proposal** flags show strong positive association with winning
- **Competitor count** is negatively correlated with win probability
- **Referral** and **Inbound** leads have higher win rates than Outbound
- **Rep win rate history** and **quota attainment** are meaningful predictors
- **Deal size** follows a log-normal distribution — log transform recommended
- **Existing customers** win at a notably higher rate than new prospects